# EWS Fraud Detection - Tahap 3A: Panel Data Validation, Penanganan Outlier, & Perhitungan Komponen Beneish M-Score

Notebook ini memproses metrik keuangan terfokus yang dihasilkan dari Tahap 2B (`Financial_Metrics_Focused.xlsx`) dengan fitur lanjutan:
1. **Panel Data Validation**: Verifikasi struktur data panel (Emiten dan Tahun) dan mengidentifikasi tahun observasi per emiten.
2. **Penyaringan Perusahaan Dormant (Dormancy Screener)**:
   - Mengidentifikasi perusahaan yang hampir tidak beroperasi (dormant) dengan kriteria: **Revenue < 1% dari Total Assets**.
   - Perusahaan yang dormant pada tahun berjalan atau tahun sebelumnya disaring agar tidak merusak perhitungan rasio keuangan (mencegah pembagi mendekati nol).
3. **Feature Engineering (Lag Variables)**: Membuat variabel lag ($t-1$) untuk:
   - `revenue`
   - `receivables`
   - `total_assets`
   - `current_assets`
   - `ppe`
   - `gross_profit`
   - `total_liabilities`
4. **Penanganan Outlier & Pembagi Nol (Capping & Division Check)**:
   - Menggunakan pembagian aman (`safe_divide`) untuk mendeteksi pembagi nol dan mengembalikannya sebagai `NaN`.
   - Menyimpan nilai mentah (`_raw`) dan nilai bersih yang dibatasi (**capped** antara `0.1` dan `10.0`) untuk mencegah outlier ekstrem (misalnya SGI melonjak hingga ribuan karena penjualan melonjak dari angka sangat kecil).
5. **Perhitungan Komponen Beneish M-Score**:
   - **DSRI** (Days Sales in Receivables Index)
   - **GMI** (Gross Margin Index)
   - **AQI** (Asset Quality Index)
   - **SGI** (Sales Growth Index)
   - **LVGI** (Leverage Index)

In [7]:
import os
import pandas as pd
import numpy as np

focused_path = "Financial_Metrics_Focused.xlsx"
out_path = "Financial_Metrics_With_Components.xlsx"

## 1. Panel Data Validation
Memuat data, mengurutkannya secara tepat berdasarkan `kode` (emiten) dan `tahun` (periode) secara menaik, serta melakukan analisis ketersediaan data panel.

In [8]:
if not os.path.exists(focused_path):
    print(f"ERROR: File {focused_path} tidak ditemukan!")
else:
    df = pd.read_excel(focused_path)
    
    # Pengurutan panel yang ketat
    df = df.sort_values(['kode', 'tahun']).reset_index(drop=True)
    
    print("=== PANEL DATA VALIDATION ===")
    print(f"Total baris data: {len(df)}")
    print(f"Jumlah emiten unik (kode): {df['kode'].nunique()}")
    print(f"Distribusi tahun:\n{df['tahun'].value_counts().sort_index()}")
    
    # Hitung jumlah tahun ketersediaan data per emiten
    counts = df['kode'].value_counts()
    print(f"\nEmiten dengan data lengkap 4 tahun: {sum(counts == 4)}")
    print(f"Emiten dengan data 3 tahun: {sum(counts == 3)}")
    print(f"Emiten dengan data 2 tahun: {sum(counts == 2)}")
    print(f"Emiten dengan data 1 tahun: {sum(counts == 1)}")

=== PANEL DATA VALIDATION ===
Total baris data: 2466
Jumlah emiten unik (kode): 629
Distribusi tahun:
tahun
2021    609
2022    622
2023    617
2024    618
Name: count, dtype: int64

Emiten dengan data lengkap 4 tahun: 589
Emiten dengan data 3 tahun: 34
Emiten dengan data 2 tahun: 2
Emiten dengan data 1 tahun: 4


## 2. Feature Engineering: Lag Variables ($t-1$)
Membuat variabel pembanding periode sebelumnya untuk variabel yang dibutuhkan.

In [9]:
# Variabel yang membutuhkan lag
cols_to_lag = [
    'revenue',
    'receivables',
    'total_assets',
    'current_assets',
    'ppe',
    'gross_profit',
    'total_liabilities'
]

for col in cols_to_lag:
    # Melakukan shift(1) per kelompok emiten (kode)
    df[f'{col}_lag'] = df.groupby('kode')[col].shift(1)

print("Variabel Lag (t-1) berhasil dibuat untuk 7 kolom utama.")

Variabel Lag (t-1) berhasil dibuat untuk 7 kolom utama.


## 3. Penanganan Outlier & Penyaringan Perusahaan Dormant
Mengidentifikasi emiten yang memiliki aktivitas bisnis sangat kecil (pendapatan < 1% dari total aset) untuk menyaring distorsi rasio.

In [10]:
def safe_divide(num, denom):
    """
    Melakukan pembagian aman. Jika penyebut (denominator) bernilai 0 atau NaN,
    atau tidak terhingga, akan mengembalikan NaN.
    """
    with np.errstate(divide='ignore', invalid='ignore'):
        res = num / denom
        res = np.where(np.isfinite(res), res, np.nan)
    return res

# Deteksi Dormancy (Pendapatan < 1% Aset)
df['dormant'] = safe_divide(df['revenue'], df['total_assets']) < 0.01
df['dormant_lag'] = safe_divide(df['revenue_lag'], df['total_assets_lag']) < 0.01

print(f"Jumlah perusahaan terdeteksi dormant pada tahun berjalan: {df['dormant'].sum()}")
print(f"Jumlah perusahaan terdeteksi dormant pada tahun sebelumnya: {df['dormant_lag'].sum()}")

Jumlah perusahaan terdeteksi dormant pada tahun berjalan: 50
Jumlah perusahaan terdeteksi dormant pada tahun sebelumnya: 38


## 4. Perhitungan Komponen Beneish M-Score dengan Capping
Menghitung rasio keuangan mentah (`_raw`) dan rasio bersih yang dibatasi (**capped** antara `0.1` dan `10.0`) untuk meredam outlier ekstrem.

In [11]:
print("=== MENGHITUNG KOMPONEN BENEISH ===")

# 1. DSRI (Days Sales in Receivables Index)
ratio_rec_t = safe_divide(df['receivables'], df['revenue'])
ratio_rec_lag = safe_divide(df['receivables_lag'], df['revenue_lag'])
# Set NaN jika dormant untuk menghindari distorsi pembagian angka kecil
ratio_rec_t = np.where(df['dormant'], np.nan, ratio_rec_t)
ratio_rec_lag = np.where(df['dormant_lag'], np.nan, ratio_rec_lag)
df['dsri_raw'] = safe_divide(ratio_rec_t, ratio_rec_lag)
df['dsri'] = df['dsri_raw'].clip(lower=0.1, upper=10.0)

# 2. GMI (Gross Margin Index)
margin_t = safe_divide(df['gross_profit'], df['revenue'])
margin_lag = safe_divide(df['gross_profit_lag'], df['revenue_lag'])
margin_t = np.where(df['dormant'], np.nan, margin_t)
margin_lag = np.where(df['dormant_lag'], np.nan, margin_lag)
df['gmi_raw'] = safe_divide(margin_lag, margin_t)
df['gmi'] = df['gmi_raw'].clip(lower=0.1, upper=10.0)

# 3. AQI (Asset Quality Index)
aq_ratio_t = 1.0 - safe_divide(df['current_assets'] + df['ppe'], df['total_assets'])
aq_ratio_lag = 1.0 - safe_divide(df['current_assets_lag'] + df['ppe_lag'], df['total_assets_lag'])
df['aqi_raw'] = safe_divide(aq_ratio_t, aq_ratio_lag)
df['aqi'] = df['aqi_raw'].clip(lower=0.1, upper=10.0)

# 4. SGI (Sales Growth Index)
# Jika lag revenue dormant, saring untuk menghindari SGI tak terhingga
sgi_rev_lag = np.where(df['dormant_lag'], np.nan, df['revenue_lag'])
df['sgi_raw'] = safe_divide(df['revenue'], sgi_rev_lag)
df['sgi'] = df['sgi_raw'].clip(lower=0.1, upper=10.0)

# 5. LVGI (Leverage Index)
lev_t = safe_divide(df['total_liabilities'], df['total_assets'])
lev_lag = safe_divide(df['total_liabilities_lag'], df['total_assets_lag'])
df['lvgi_raw'] = safe_divide(lev_t, lev_lag)
df['lvgi'] = df['lvgi_raw'].clip(lower=0.1, upper=10.0)

print("Komponen Beneish berhasil dihitung dengan filter outlier.")

# Evaluasi ketersediaan kolom capped
indices = ['dsri', 'gmi', 'aqi', 'sgi', 'lvgi']
for idx in indices:
    non_null = df[idx].notnull().sum()
    pct = (non_null / len(df)) * 100
    print(f"- {idx.upper()}: {non_null} non-null ({pct:.2f}% dari total baris)")

=== MENGHITUNG KOMPONEN BENEISH ===
Komponen Beneish berhasil dihitung dengan filter outlier.
- DSRI: 1733 non-null (70.28% dari total baris)
- GMI: 1698 non-null (68.86% dari total baris)
- AQI: 1806 non-null (73.24% dari total baris)
- SGI: 1795 non-null (72.79% dari total baris)
- LVGI: 1835 non-null (74.41% dari total baris)


## 5. Eksplorasi & Ekspor Hasil
Menampilkan contoh hasil perhitungan untuk emiten AALI dan menyimpan dataset panel lengkap ke format Excel.

In [12]:
print("=== Contoh Hasil Perhitungan AALI (Cleaned vs Raw) ===")
columns_to_show = [
    'kode', 'tahun', 
    'dsri_raw', 'dsri', 
    'gmi_raw', 'gmi',
    'sgi_raw', 'sgi'
]
print(df[df['kode'] == 'AALI'][columns_to_show].to_string(index=False))

# Ekspor
df.to_excel(out_path, index=False)
print(f"\nSukses menyimpan data ke '{out_path}'. Bentuk tabel: {df.shape}")

=== Contoh Hasil Perhitungan AALI (Cleaned vs Raw) ===
kode  tahun  dsri_raw     dsri  gmi_raw      gmi  sgi_raw      sgi
AALI   2021       NaN      NaN      NaN      NaN      NaN      NaN
AALI   2022  2.285853 2.285853 1.134149 1.134149 0.897482 0.897482
AALI   2023  0.304869 0.304869 1.310896 1.310896 0.950381 0.950381
AALI   2024  1.385800 1.385800 0.872246 0.872246 1.051556 1.051556

Sukses menyimpan data ke 'Financial_Metrics_With_Components.xlsx'. Bentuk tabel: (2466, 36)
